Оценка одной RAG-системы через RAGAS.

Один переключатель `LLM_BACKEND` управляет и генерацией в проверяемой RAG-системе, и judge-моделью RAGAS.

In [1]:
import os
from pathlib import Path

import pandas as pd

from langchain_community.embeddings import HuggingFaceEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.run_config import RunConfig

from llm_interface import build_langchain_chat_model
from rag_eval.run_eval import LangchainEmbeddingsWrapper, latest_xlsx, run_single_rag_eval
from rag_systems.gigachat_bge_m3 import RAG_SYSTEM_NAME, build_rag_system

# ===== Пути и входные данные =====
# DATA_DIR: папка с золотыми вопросами (.xlsx).
DATA_DIR = Path('data')
# REPORTS_DIR: куда сохраняются все артефакты оценки (scores, summary, report.html).
REPORTS_DIR = Path('reports')
# GOLD_XLSX: автоматически выбирается самый свежий .xlsx из DATA_DIR.
GOLD_XLSX = latest_xlsx(DATA_DIR)
# GOLD_LIMIT: ограничить число строк для быстрого прогона (0 или <0 = без ограничения).
GOLD_LIMIT = int(os.environ.get('GOLD_LIMIT', '2'))
if GOLD_LIMIT > 0:
    _gold_df = pd.read_excel(GOLD_XLSX).head(GOLD_LIMIT)
    GOLD_XLSX = DATA_DIR / f'{GOLD_XLSX.stem}_top{GOLD_LIMIT}.xlsx'
    _gold_df.to_excel(GOLD_XLSX, index=False)

# ===== Выбор backend =====
# Единый backend для генерации RAG-ответа и judge-оценки RAGAS: 'koboldcpp' или 'gigachat'.
LLM_BACKEND = os.environ.get('LLM_BACKEND', 'koboldcpp')
# Если 1, принудительно разрешен только локальный backend koboldcpp.
ISOLATED_LOCAL_ONLY = os.environ.get('ISOLATED_LOCAL_ONLY', '1') == '1'
if ISOLATED_LOCAL_ONLY and LLM_BACKEND != 'koboldcpp':
    raise ValueError("ISOLATED_LOCAL_ONLY=1 разрешает только LLM_BACKEND='koboldcpp'")

# ===== Таймауты и параллелизм =====
# Таймаут одного запроса judge-LLM (сек). Для изолированной среды можно поставить 5.
JUDGE_TIMEOUT_SECONDS = float(os.environ.get('JUDGE_TIMEOUT_SECONDS', '0'))
# Таймаут задачи RAGAS целиком (сек) для одной строки/метрики.
RAGAS_TIMEOUT_SECONDS = float(os.environ.get('RAGAS_TIMEOUT_SECONDS', '1800'))
# Число параллельных воркеров RAGAS (1 = последовательный режим, наиболее стабильный).
RAGAS_MAX_WORKERS = int(os.environ.get('RAGAS_MAX_WORKERS', '1'))

# ===== Локальные эмбеддинги для judge-метрик =====
# Путь к локальной embedding-модели (например bge-m3).
LOCAL_EMBED_MODEL_PATH = os.environ.get('LOCAL_EMBED_MODEL_PATH', '/home/vladislav/models/bge-m3')
# Устройство для эмбеддингов: 'cpu' или 'cuda'.
LOCAL_EMBED_DEVICE = os.environ.get('LOCAL_EMBED_DEVICE', os.environ.get('EMBEDDING_DEVICE', 'cpu'))
# Таймаут LLM самой RAG-системы (используется в rag_systems/gigachat_bge_m3.py).
os.environ.setdefault('MODEL_TIMEOUT_SECONDS', '0')

# ===== Judge LLM (для ragas.evaluate) =====
judge_chat_model = build_langchain_chat_model(
    backend=LLM_BACKEND,
    timeout=JUDGE_TIMEOUT_SECONDS,
    gigachat_options={
        # Параметры GigaChat API (актуальны при LLM_BACKEND='gigachat').
        'model': os.environ.get('GIGACHAT_MODEL'),
        'credentials': os.environ.get('GIGACHAT_CREDENTIALS'),
        'scope': os.environ.get('GIGACHAT_SCOPE'),
    },
    koboldcpp_options={
        # Параметры локального OpenAI-compatible endpoint (актуальны при LLM_BACKEND='koboldcpp').
        'base_url': os.environ.get('KOBOLDCPP_BASE_URL', 'http://127.0.0.1:5001/v1'),
        'api_key': os.environ.get('KOBOLDCPP_API_KEY', 'koboldcpp'),
        'model': os.environ.get('KOBOLDCPP_MODEL', 'koboldcpp'),
        'temperature': 0.0,
    },
)
judge_llm = LangchainLLMWrapper(judge_chat_model)

# Judge embeddings нужны для метрик типа answer_relevancy.
judge_embeddings = None
if LangchainEmbeddingsWrapper is not None:
    try:
        local_embeddings = HuggingFaceEmbeddings(
            model_name=LOCAL_EMBED_MODEL_PATH,
            model_kwargs={'device': LOCAL_EMBED_DEVICE},
            encode_kwargs={'normalize_embeddings': True},
        )
        judge_embeddings = LangchainEmbeddingsWrapper(local_embeddings)
    except Exception as exc:
        print('Local judge embeddings disabled:', exc)

# Конфиг выполнения RAGAS: timeout/parallelism.
ragas_run_config = RunConfig(timeout=RAGAS_TIMEOUT_SECONDS, max_workers=RAGAS_MAX_WORKERS)

# ===== Проверяемая RAG-система =====
# build_rag_system берет retrieval/LLM настройки из rag_systems/gigachat_bge_m3.py и env-переменных.
rag_system = build_rag_system(llm_backend=LLM_BACKEND)
print('Backend:', LLM_BACKEND)
print('Judge timeout (s):', JUDGE_TIMEOUT_SECONDS)
print('RAGAS timeout (s):', RAGAS_TIMEOUT_SECONDS)
print('Judge embeddings local:', judge_embeddings is not None)



/home/vladislav/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_136830/548799221.py:52: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(judge_chat_model)
/tmp/ipykernel_136830/548799221.py:56: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import Huggi

Backend: koboldcpp
Judge timeout (s): 0.0
RAGAS timeout (s): 1800.0
Judge embeddings local: True


In [2]:
# Запуск оценки одной RAG-системы.
run_dir = run_single_rag_eval(
    # Путь к xlsx с вопросами/эталонами.
    gold_path=GOLD_XLSX,
    # Экземпляр тестируемой RAG-системы (генерация + retrieval).
    rag_system=rag_system,
    # Judge LLM для LLM-метрик RAGAS.
    judge_llm=judge_llm,
    # Judge embeddings для embedding-метрик (если доступны).
    judge_embeddings=judge_embeddings,
    # RunConfig для таймаутов и параллелизма RAGAS.
    ragas_run_config=ragas_run_config,
    # Корневая папка, куда будет сохранен run.
    reports_dir=REPORTS_DIR,
    # Имя системы в названии папки отчета.
    run_name=RAG_SYSTEM_NAME,
)

print('Saved:', run_dir.resolve())



Evaluating:  75%|███████▌  | 3/4 [08:59<03:28, 208.58s/it]Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt context_recall_classification_prompt failed to parse output: The output parser failed to parse the output including retries.
Exception raised in Job[3]: RagasOutputParserException(The output parser failed to parse the output including retries.)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 13664.16it/s]


Saved: /home/vladislav/Рабочий стол/working/ragas/reports/2026-03-09_22-03-36_gigachat_bge_m3
